# 面试问题：HNSW 如何搜索，过滤条件为什么不能阻断图遍历？

可以直接复述的回答是：HNSW 用稀疏多层近邻图近似小世界导航，高层做长距离 greedy descent，底层用 `efSearch` 候选队列扩展。访问成本取决于图连通性、入口点、M 与 efSearch，而不是固定扫描全库。业务过滤通常区分“能否作为结果”和“能否作为路由节点”；一个下架或闭店节点仍可能是连接两个区域的桥。若在遍历时直接删掉过滤节点，图会断开并漏召回。正确做法通常是允许其参与导航、只在最终结果中排除，同时监控删除比例并重建图。下面用十二家餐厅的分层邻接表手写搜索。

## 真实案例：带营业状态过滤的餐厅向量召回

十二家餐厅沿两片商圈分布，R-06 是已经闭店的交通枢纽餐厅，也是图上的桥节点。二维坐标是教学 embedding 投影，不是经纬度；邻接表保留 HNSW 的层级和稀疏长边结构。

In [1]:
import heapq  # 导入最小堆实现 efSearch 候选队列
import numpy as np  # 导入二维向量距离计算
restaurants = [  # 定义十二家带营业状态的餐厅向量
    ("R-01", "西站面馆", [0.0, 0.1], True),  # 第一商圈入口节点
    ("R-02", "河畔咖啡", [0.9, 0.0], True),  # 第一商圈咖啡店
    ("R-03", "老街火锅", [1.8, 0.2], True),  # 第一商圈火锅店
    ("R-04", "创意寿司", [2.7, 0.1], True),  # 第一商圈日料店
    ("R-05", "广场轻食", [3.6, 0.0], True),  # 第一商圈边缘店
    ("R-06", "枢纽餐厅", [4.6, 0.1], False),  # 已闭店但连接两个商圈的桥
    ("R-07", "东城烧烤", [5.6, 0.0], True),  # 第二商圈入口店
    ("R-08", "湖景餐吧", [6.5, 0.2], True),  # 第二商圈餐吧
    ("R-09", "科技园简餐", [7.4, 0.1], True),  # 第二商圈工作餐
    ("R-10", "东站咖啡", [8.3, 0.0], True),  # 第二商圈咖啡店
    ("R-11", "会展牛排", [9.2, 0.2], True),  # 第二商圈牛排店
    ("R-12", "机场面馆", [10.1, 0.1], True),  # 最远端面馆
]  # 结束十二家餐厅
vectors = {row[0]: np.asarray(row[2], dtype=np.float64) for row in restaurants}  # 建立节点到二维向量映射
names = {row[0]: row[1] for row in restaurants}  # 建立节点到餐厅名称映射
open_status = {row[0]: row[3] for row in restaurants}  # 建立营业过滤字段映射
queries = [  # 定义六个附近餐厅查询向量
    ("西站早餐", np.array([0.2, 0.1])),  # 接近 R-01 的查询
    ("老街聚餐", np.array([1.9, 0.1])),  # 接近 R-03 的查询
    ("广场轻食", np.array([3.5, 0.1])),  # 接近 R-05 的查询
    ("东城晚餐", np.array([5.7, 0.1])),  # 需要跨过闭店桥节点
    ("东站咖啡", np.array([8.2, 0.0])),  # 第二商圈深处查询
    ("机场夜宵", np.array([10.0, 0.1])),  # 最远端查询
]  # 结束六个查询
print("输入预览：node | name | vector | open")  # 输出餐厅图节点表头
for node_id, name, vector, is_open in restaurants:  # 逐条展示十二个节点
    print(f"{node_id} | {name:6} | {vector} | {is_open}")  # 展示业务属性和过滤状态
print("查询数：", len(queries), "入口节点：R-01")  # 展示搜索请求规模和图入口

输入预览：node | name | vector | open
R-01 | 西站面馆   | [0.0, 0.1] | True
R-02 | 河畔咖啡   | [0.9, 0.0] | True
R-03 | 老街火锅   | [1.8, 0.2] | True
R-04 | 创意寿司   | [2.7, 0.1] | True
R-05 | 广场轻食   | [3.6, 0.0] | True
R-06 | 枢纽餐厅   | [4.6, 0.1] | False
R-07 | 东城烧烤   | [5.6, 0.0] | True
R-08 | 湖景餐吧   | [6.5, 0.2] | True
R-09 | 科技园简餐  | [7.4, 0.1] | True
R-10 | 东站咖啡   | [8.3, 0.0] | True
R-11 | 会展牛排   | [9.2, 0.2] | True
R-12 | 机场面馆   | [10.1, 0.1] | True
查询数： 6 入口节点：R-01


## Baseline / 基线：对全部营业餐厅做 Exact Search

精确基线扫描十二个向量，但只允许营业节点进入结果，用作 HNSW top1 召回的权威答案。

In [2]:
def distance(query, node_id):  # 计算查询到指定图节点的平方欧氏距离
    return float(((query - vectors[node_id]) ** 2).sum())  # 返回无需开方的排序等价距离
def exact_open_search(query):  # 实现带营业过滤的全量搜索
    scored = [(node_id, distance(query, node_id)) for node_id in vectors if open_status[node_id]]  # 只对营业节点生成结果分数
    scored.sort(key=lambda item: (item[1], item[0]))  # 按距离和节点 ID 稳定排序
    return scored[0], len(vectors)  # 返回最近营业餐厅和全库扫描数
print("query | exact_open_top1 | distance | scanned")  # 输出精确检索结果表头
for query_name, query_vector in queries:  # 遍历六个餐厅请求
    result, scanned = exact_open_search(query_vector)  # 获取权威最近营业节点
    print(f"{query_name:6} | {result[0]} {names[result[0]]} | {result[1]:.4f} | {scanned}")  # 展示同数据基线结果

query | exact_open_top1 | distance | scanned
西站早餐   | R-01 西站面馆 | 0.0400 | 12
老街聚餐   | R-03 老街火锅 | 0.0200 | 12
广场轻食   | R-05 广场轻食 | 0.0200 | 12
东城晚餐   | R-07 东城烧烤 | 0.0200 | 12
东站咖啡   | R-10 东站咖啡 | 0.0100 | 12
机场夜宵   | R-12 机场面馆 | 0.0100 | 12


## 核心实现：三层稀疏图、greedy descent 与底层 efSearch

高层保留长距离边，底层保留局部链和少量 shortcut。邻接表是可直接审计的 HNSW 索引产物；查询代码从入口逐层下降，并记录每个访问节点。

In [3]:
layers = {  # 定义三层 HNSW 教学邻接表
    2: {"R-01": ["R-06"], "R-06": ["R-01", "R-10"], "R-10": ["R-06"]},  # 顶层跨商圈高速边
    1: {"R-01": ["R-03"], "R-03": ["R-01", "R-06"], "R-06": ["R-03", "R-08"], "R-08": ["R-06", "R-10"], "R-10": ["R-08", "R-12"], "R-12": ["R-10"]},  # 中层稀疏导航边
    0: {},  # 底层邻接表随后按局部链构建
}  # 结束多层图定义
for index, row in enumerate(restaurants):  # 遍历十二个底层节点
    node_id = row[0]  # 读取当前节点身份
    neighbors = []  # 收集当前节点链式和跳跃邻居
    if index > 0:  # 检查是否存在左侧相邻节点
        neighbors.append(restaurants[index - 1][0])  # 添加左侧局部近邻
    if index + 1 < len(restaurants):  # 检查是否存在右侧相邻节点
        neighbors.append(restaurants[index + 1][0])  # 添加右侧局部近邻
    if index + 2 < len(restaurants) and node_id != "R-05":  # 增加小范围 shortcut 但不绕过唯一闭店桥节点
        neighbors.append(restaurants[index + 2][0])  # 添加跨一个节点的前向边
    layers[0][node_id] = neighbors  # 写入当前底层邻接列表
def greedy_descent(query, entry, layer, expand_filtered, trace):  # 在指定高层执行贪心下降
    current = entry  # 从上一层入口开始导航
    improved = True  # 初始化继续寻找更近邻居的标记
    while improved:  # 只要存在更近邻居就继续移动
        improved = False  # 默认本轮不移动
        current_distance = distance(query, current)  # 计算当前入口距离
        for neighbor in layers[layer].get(current, []):  # 遍历当前节点的层内邻居
            trace.append((layer, neighbor, distance(query, neighbor), open_status[neighbor]))  # 记录实际访问节点、距离和状态
            if not expand_filtered and not open_status[neighbor]:  # 错误策略不允许闭店节点参与路由
                continue  # 跳过闭店桥节点及其后续边
            neighbor_distance = distance(query, neighbor)  # 读取可导航邻居距离
            if neighbor_distance < current_distance:  # 检查邻居是否更接近查询
                current = neighbor  # 将更近邻居作为新入口
                improved = True  # 标记需要继续本层搜索
                break  # 从新节点重新检查邻接表
    return current  # 返回本层最终入口
def hnsw_search(query, ef_search=4, expand_filtered=True):  # 手写高层导航和底层 best-first 搜索
    trace = [(2, "R-01", distance(query, "R-01"), open_status["R-01"])]  # 从固定顶层入口初始化访问轨迹
    entry = greedy_descent(query, "R-01", 2, expand_filtered, trace)  # 在顶层执行长距离导航
    entry = greedy_descent(query, entry, 1, expand_filtered, trace)  # 在中层进一步细化入口
    candidate_heap = [(distance(query, entry), entry)]  # 用距离最小堆初始化底层候选队列
    visited = {entry}  # 记录已加入队列的节点避免重复访问
    explored = []  # 保存底层弹出的候选节点
    while candidate_heap and len(explored) < ef_search:  # 最多扩展 efSearch 个底层节点
        current_distance, current = heapq.heappop(candidate_heap)  # 取出当前最近候选
        explored.append((current, current_distance, open_status[current]))  # 记录扩展节点和过滤状态
        if not expand_filtered and not open_status[current]:  # 错误策略在闭店节点处停止展开
            continue  # 不访问桥节点的邻居
        for neighbor in layers[0].get(current, []):  # 遍历底层局部和 shortcut 邻居
            if neighbor in visited:  # 检查邻居是否已入队
                continue  # 跳过重复节点
            visited.add(neighbor)  # 标记新邻居已经发现
            if not expand_filtered and not open_status[neighbor]:  # 错误策略连闭店邻居也不入队
                continue  # 阻断穿越过滤节点的路径
            heapq.heappush(candidate_heap, (distance(query, neighbor), neighbor))  # 按查询距离加入候选堆
    open_results = [(node_id, node_distance) for node_id, node_distance, is_open in explored if is_open]  # 只让营业节点进入最终结果
    open_results.sort(key=lambda item: (item[1], item[0]))  # 按距离稳定排序营业候选
    return open_results, trace, explored, visited  # 返回结果、高层轨迹、底层扩展和访问集合
sample_results, sample_trace, sample_explored, sample_visited = hnsw_search(queries[4][1], 6, True)  # 对东站查询执行完整可导航过滤搜索
print("东站查询高层访问：layer | node | distance | open")  # 输出高层导航轨迹表头
for row in sample_trace:  # 遍历顶层和中层访问记录
    print(row)  # 展示长距离边如何接近第二商圈
print("底层弹出顺序：", sample_explored)  # 展示 efSearch 候选扩展过程
print("最终营业候选：", sample_results)  # 展示过滤只应用于结果

东站查询高层访问：layer | node | distance | open
(2, 'R-01', 67.25, True)
(2, 'R-06', 12.969999999999997, False)
(2, 'R-01', 67.25, True)
(2, 'R-10', 0.010000000000000285, True)
(2, 'R-06', 12.969999999999997, False)
(1, 'R-08', 2.9299999999999975, True)
(1, 'R-12', 3.620000000000001, True)
底层弹出顺序： [('R-10', 0.010000000000000285, True), ('R-09', 0.6499999999999984, True), ('R-11', 1.04, True), ('R-08', 2.9299999999999975, True), ('R-12', 3.620000000000001, True), ('R-07', 6.759999999999998, True)]
最终营业候选： [('R-10', 0.010000000000000285), ('R-09', 0.6499999999999984), ('R-11', 1.04), ('R-08', 2.9299999999999975), ('R-12', 3.620000000000001), ('R-07', 6.759999999999998)]


## 六查询结果与访问成本

In [4]:
result_rows = []  # 收集六个查询的 HNSW 评估结果
print("query | exact | hnsw | hit | visited | full_scan")  # 输出近似与精确对照表头
for query_name, query_vector in queries:  # 遍历六个餐厅查询
    exact_result, scanned = exact_open_search(query_vector)  # 获取全扫描权威最近邻
    approximate, trace, explored, visited = hnsw_search(query_vector, 6, True)  # 执行允许过滤节点导航的 HNSW
    top_id = approximate[0][0] if approximate else None  # 提取 HNSW 最近营业结果
    hit = top_id == exact_result[0]  # 判断近似 top1 是否命中精确答案
    result_rows.append((query_name, exact_result[0], top_id, hit, len(visited), scanned))  # 保存召回和访问成本
    print(f"{query_name:6} | {exact_result[0]} | {top_id} | {hit} | {len(visited):7d} | {scanned}")  # 展示逐查询结果
hnsw_recall = sum(row[3] for row in result_rows) / len(result_rows)  # 计算六查询 top1 recall
average_visited = sum(row[4] for row in result_rows) / len(result_rows)  # 计算平均发现节点数
print(f"教学实验 recall@1={hnsw_recall:.1%}，平均发现节点={average_visited:.1f}/{len(restaurants)}")  # 汇总效果与成本

query | exact | hnsw | hit | visited | full_scan
西站早餐   | R-01 | R-01 | True |       8 | 12
老街聚餐   | R-03 | R-03 | True |       8 | 12
广场轻食   | R-05 | R-05 | True |       9 | 12
东城晚餐   | R-07 | R-07 | True |       9 | 12
东站咖啡   | R-10 | R-10 | True |       7 | 12
机场夜宵   | R-12 | R-12 | True |       7 | 12
教学实验 recall@1=100.0%，平均发现节点=8.0/12


## 失败案例与修正：闭店桥节点被当作不可遍历

对“东站咖啡”查询，错误策略在高层遇到 R-06 就跳过，只能停留在第一商圈。修正策略仍扩展 R-06，但最终结果过滤它，因此既满足营业约束又保持图连通。

In [5]:
failure_query = queries[4][1]  # 选择需要跨商圈导航的东站查询
naive_results, naive_trace, naive_explored, naive_visited = hnsw_search(failure_query, 6, False)  # 复现过滤节点不可遍历的错误实现
fixed_results, fixed_trace, fixed_explored, fixed_visited = hnsw_search(failure_query, 6, True)  # 执行可导航但不可返回的修正实现
exact_failure, failure_scan = exact_open_search(failure_query)  # 获取该查询的精确营业近邻
naive_top = naive_results[0][0] if naive_results else None  # 提取错误策略第一名
fixed_top = fixed_results[0][0] if fixed_results else None  # 提取修正策略第一名
print("错误策略高层访问：", naive_trace)  # 展示闭店桥被跳过后的轨迹
print("错误策略底层扩展：", naive_explored, "top=", naive_top)  # 展示搜索困在第一商圈
print("修正策略高层访问：", fixed_trace)  # 展示通过 R-06 抵达第二商圈
print("修正策略底层扩展：", fixed_explored, "top=", fixed_top)  # 展示营业结果与桥节点分离
print("精确营业 top1：", exact_failure)  # 展示修正方案命中权威答案

错误策略高层访问： [(2, 'R-01', 67.25, True), (2, 'R-06', 12.969999999999997, False), (1, 'R-03', 40.99999999999999, True), (1, 'R-01', 67.25, True), (1, 'R-06', 12.969999999999997, False)]
错误策略底层扩展： [('R-03', 40.99999999999999, True), ('R-05', 21.159999999999997, True), ('R-04', 30.25999999999999, True), ('R-02', 53.289999999999985, True), ('R-01', 67.25, True)] top= R-05
修正策略高层访问： [(2, 'R-01', 67.25, True), (2, 'R-06', 12.969999999999997, False), (2, 'R-01', 67.25, True), (2, 'R-10', 0.010000000000000285, True), (2, 'R-06', 12.969999999999997, False), (1, 'R-08', 2.9299999999999975, True), (1, 'R-12', 3.620000000000001, True)]
修正策略底层扩展： [('R-10', 0.010000000000000285, True), ('R-09', 0.6499999999999984, True), ('R-11', 1.04, True), ('R-08', 2.9299999999999975, True), ('R-12', 3.620000000000001, True), ('R-07', 6.759999999999998, True)] top= R-10
精确营业 top1： ('R-10', 0.010000000000000285)


## 结果解读

高层长边让东站查询从 R-01 快速接近 R-10，底层 efSearch 再比较局部节点。闭店 R-06 从未进入最终结果，但它必须保留导航能力；把业务过滤同时用于图展开会制造不可见的连通性故障。

## 生产边界

教学图使用人工可审计邻接表和二维向量，没有真实 HNSW 插入、邻居多样性裁剪、并发更新或压缩存储。生产中 M、efConstruction 与 efSearch 要用 exact holdout 调参；删除节点比例过高应重建图。ACL 等敏感过滤还需保证候选不足时不泄露禁止节点信息，并监控分段召回与访问跳数。

## 最小回归测试

In [6]:
assert len(restaurants) >= 6 and len(queries) >= 6  # 保证案例覆盖多个图节点与查询
assert open_status["R-06"] is False  # 保证失败案例桥节点确实处于过滤状态
assert naive_top != exact_failure[0]  # 保证不可遍历过滤真实导致漏召回
assert fixed_top == exact_failure[0]  # 保证导航与返回过滤分离后找回精确近邻
assert all(open_status[node_id] for node_id, node_distance in fixed_results)  # 保证修正结果中不返回闭店节点
assert hnsw_recall == 1.0  # 保证六个教学查询均命中精确 top1
assert average_visited < len(restaurants)  # 保证平均发现节点少于全量扫描